<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/SLM-Finetuning-Demo/blob/main/SLM-Finetuning-without-Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ================================
# Install Required Libraries
# ================================
!pip -q install transformers datasets peft accelerate bitsandbytes trl

# ================================
# Imports
# ================================
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 43.5 MB/s eta 0:00:00


In [29]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [30]:

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

In [31]:

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # fix PAD token warning

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # FP16, NOT BF16
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [19]:

from torch.cuda.amp import autocast

# Disable mixed precision entirely for 4-bit + LoRA
training_args.fp16 = False
training_args.bf16 = False

In [32]:

# prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# attach LoRA adapters
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# disable caching (required for gradient checkpointing)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# force FP16
model.half()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [34]:

dataset = load_dataset("json", data_files="arise_finetune_dataset_2196.json")

# combine instruction, input, output into single text
def format_prompt(example):
    if example["input"].strip() != "":
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    return {"text": text}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [35]:

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/2196 [00:00<?, ? examples/s]

In [39]:

training_args = TrainingArguments(
    output_dir="./slm-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=200,

    fp16=False,  # do NOT use bf16 or fp16 auto — manual FP16 is used
    bf16=False,

    optim="paged_adamw_8bit"
)

In [40]:

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args
)

In [41]:
trainer.train()

Step,Training Loss
10,0.025127
20,0.021274
30,0.022999
40,0.020824
50,0.020144
60,0.018218
70,0.017913
80,0.016836
90,0.016397
100,0.015776


TrainOutput(global_step=275, training_loss=0.01668983502821489, metrics={'train_runtime': 932.3289, 'train_samples_per_second': 2.355, 'train_steps_per_second': 0.295, 'total_flos': 6994134048964608.0, 'train_loss': 0.01668983502821489})